## Imports and Loading

In [2]:
import numpy as np
import pandas as pd

In [3]:
import re # for regex matching

In [4]:
df = pd.read_csv("./data/raw.csv")

print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded dataset: 103 rows, 11 columns


## Overview

In [5]:
df.head(10)

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,11/22/2024,Blender,Home,3,38,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,7/5/2025,Smartphone,Electronics,2,abd,PayPal,Processing,NaN
2,102,Customer_102,ORD-84355,12/23/2024,Tennis Racket,Sports,1,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,3/19/2025,Science,Books,5,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,10/20/2025,Biography,Books,1,552.51,Cash on Delivery,Processing,552.51
5,105,Customer_105,ORD-22442,11/20/2024,Tennis Racket,Sports,3,122.06,Cash on Delivery,Cancelled,366.18
6,106,Customer_106,ORD-25885,2/2/2025,Blender,Home,NaN,978.63,Bank Transfer,Processing,NaN
7,107,Customer_107,ORD-89122,1/3/2025,Biography,Books,5,587.68,PayPal,Returned,2938.40
8,108,Customer_108,ORD-64400,10/23/2025,Science,Books,1,600.29,Cash on Delivery,Processing,600.29
9,109,Customer_109,ORD-18512,5/3/2025,Tennis Racket,Sports,5,168.34,Credit Card,Shipped,841.70


In [6]:
print("Initial columns:", df.columns.tolist())
df.columns = df.columns.str.strip()
print("Clean columns:", df.columns.tolist())

Initial columns: ['ID', ' Customer_Name', 'Order_ID', 'Order_Date', 'Product', ' Category', 'Quantity', 'Price', 'Payment_Method', 'Status', 'Total']
Clean columns: ['ID', 'Customer_Name', 'Order_ID', 'Order_Date', 'Product', 'Category', 'Quantity', 'Price', 'Payment_Method', 'Status', 'Total']


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              103 non-null    int64  
 1   Customer_Name   103 non-null    str    
 2   Order_ID        103 non-null    str    
 3   Order_Date      103 non-null    str    
 4   Product         103 non-null    str    
 5   Category        95 non-null     str    
 6   Quantity        98 non-null     str    
 7   Price           98 non-null     str    
 8   Payment_Method  103 non-null    str    
 9   Status          103 non-null    str    
 10  Total           89 non-null     float64
dtypes: float64(1), int64(1), str(9)
memory usage: 9.0 KB


Data type converions:
- `Order_Date` to datetime
- `Quantity` to int
- `Price` to float

Inter-column relations:
- `Customer_Name` = Customer_`ID` (but this is because the data is synthetic so we shouldn't be bothered here)
- $\text{Quantity} \times \text{Price} = \text{Total}$

In [8]:
df.isna().sum()

ID                 0
Customer_Name      0
Order_ID           0
Order_Date         0
Product            0
Category           8
Quantity           5
Price              5
Payment_Method     0
Status             0
Total             14
dtype: int64

## Data Type Conversion

### `Order_Date` to datetime.

In [9]:
df["Order_Date"].head(10)

0    11/22/2024
1      7/5/2025
2    12/23/2024
3     3/19/2025
4    10/20/2025
5    11/20/2024
6      2/2/2025
7      1/3/2025
8    10/23/2025
9      5/3/2025
Name: Order_Date, dtype: str

Does every value follow this `^\d{1,2}\/\d{1,2}\/\d{4}$` format (for example 11/22/2024)?

In [10]:
order_date_mask = df["Order_Date"].str.strip().str.match(r"^\d{1,2}\/\d{1,2}\/\d{4}$")

print("Non-matching values:", (~order_date_mask).sum())

display(df["Order_Date"][~order_date_mask])

Non-matching values: 2


14    Jan 5 2023
92           abc
Name: Order_Date, dtype: str

"Jan 5 2023" will automatically be converted to the majority format while "abc" will just be NaT.

In [11]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%m/%d/%Y", errors="coerce")

Now every value is in YYYY-MM-DD format.

In [12]:
display(df["Order_Date"].head())

0   2024-11-22
1   2025-07-05
2   2024-12-23
3   2025-03-19
4   2025-10-20
Name: Order_Date, dtype: datetime64[us]

### `Quantity` to int

In [13]:
df["Quantity"].head()

0    3
1    2
2    1
3    5
4    1
Name: Quantity, dtype: str

Check if every value resembles a number `\d+`.

In [14]:
quantity_mask = df["Quantity"].str.strip().str.match(r"^\d+$")

print("Non-matching values:", (~quantity_mask).sum())

display(df["Quantity"][~quantity_mask])

Non-matching values: 8


6     NaN
17     -2
34     -5
64    NaN
67    NaN
92     4a
93    NaN
96    NaN
Name: Quantity, dtype: str

We convert all these 8 values to NaN.

In [15]:
def to_int(quantity_str: str):
    if pd.isna(quantity_str):
        return pd.NA

    s = quantity_str.strip()

    if s.isdigit():
        return int(s)

    return pd.NA

df["Quantity"] = df["Quantity"].apply(to_int).astype("Int64")

display(df["Quantity"].head())

0    3
1    2
2    1
3    5
4    1
Name: Quantity, dtype: Int64

### `Price` to float

In [16]:
df["Price"].head()

0        38
1       abd
2    389.05
3    233.92
4    552.51
Name: Price, dtype: str

Check for the values that resemble an actual number `^\d+\.{0,1}\d*$`.

In [17]:
price_mask = df["Price"].str.strip().str.match(r"^\d+\.{0,1}\d*$")

print("Non-matching values:", (~price_mask).sum())

display(df["Price"][~price_mask])

Non-matching values: 10


1              abd
10    four hundred
16             NaN
20            300$
24             NaN
30             NaN
37            -100
56             NaN
83             NaN
96             abd
Name: Price, dtype: str

Values like "four hundred", "300$", and "-100" are among the non-matching values. It's tempting whether to discard them as NaN or convert them to their most probably meant value. But since they're just 3 in number, it's better to discard them and later on interpolate them. However, I think it would be better to note down them along with their indices to check them after interpolation.

|Index|Price|
|---|---|
|10|four hundred|
|20|300$|
|37|-100|

In [21]:
def to_float(price_str):
    if pd.isna(price_str):
        return pd.NA
    
    s = price_str.strip()
    price_pattern = re.compile(r'^\d+\.{0,1}\d*$')  # 275 27.5 27.

    if price_pattern.fullmatch(s):
        return float(s)
    
    return pd.NA

df["Price"] = df["Price"].apply(to_float).astype("Float64")

display(df["Price"].head())

0      38.0
1      <NA>
2    389.05
3    233.92
4    552.51
Name: Price, dtype: Float64

### Conclusion

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              103 non-null    int64         
 1   Customer_Name   103 non-null    str           
 2   Order_ID        103 non-null    str           
 3   Order_Date      101 non-null    datetime64[us]
 4   Product         103 non-null    str           
 5   Category        95 non-null     str           
 6   Quantity        95 non-null     Int64         
 7   Price           93 non-null     Float64       
 8   Payment_Method  103 non-null    str           
 9   Status          103 non-null    str           
 10  Total           89 non-null     float64       
dtypes: Float64(1), Int64(1), datetime64[us](1), float64(1), int64(1), str(6)
memory usage: 9.2 KB


In [23]:
df.isna().sum()

ID                 0
Customer_Name      0
Order_ID           0
Order_Date         2
Product            0
Category           8
Quantity           8
Price             10
Payment_Method     0
Status             0
Total             14
dtype: int64

A table showing the NA count change after data type conversion

| Column      | Old NA  | New NA    |
| ---         | ---     | ---       |
| `Order_Date`| 0       | 2         |
| `Quantity`  | 5       | 8         |
| `Price`     | 5       | 10        |
|             | **10**  | **20**    |










